# Basic NutriMatch vs All Other Diet Representations

This notebook compares the base `basic_nutrimatch` representation against every other diet feature set. It focuses on the thesis question:

> Does enhanced diet data improve prediction compared with basic NutriMatch?

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
PROJECT_ROOT = Path.cwd()
tre_root = Path("/home/ec2-user/studies/Diet_Data_Enhancement_Project/Diet_Data_Enhancement_TRE")

if not (PROJECT_ROOT / "downstream_analysis").exists() and tre_root.exists():
    PROJECT_ROOT = tre_root

COMPARISON_CSV = PROJECT_ROOT / "downstream_analysis/tasks/cvd/outputs/cvd_feature_set_comparison_memory_safe.csv"
OUT_DIR = PROJECT_ROOT / "downstream_analysis/tasks/cvd/outputs/basic_nutrimatch_vs_all"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Comparison CSV:", COMPARISON_CSV)
print("Output directory:", OUT_DIR)

if not COMPARISON_CSV.exists():
    raise FileNotFoundError(f"Missing comparison CSV: {COMPARISON_CSV}")

## Load Results

In [ ]:
raw = pd.read_csv(COMPARISON_CSV)

for col in ["r2_mean", "r2_std", "rmse_mean", "rmse_std", "pearson_r_mean", "pearson_r_std", "aligned_rows", "feature_count"]:
    if col in raw.columns:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")

if "model" in raw.columns and "ridge" in set(raw["model"].dropna()):
    df = raw[raw["model"].eq("ridge")].copy()
else:
    df = raw.copy()

print("Rows loaded:", len(df))
print("Feature sets:", sorted(df["feature_set"].dropna().unique()))
print("Targets:", sorted(df["target"].dropna().unique()))
df.head()

In [ ]:
TARGET_LABELS = {
    "bt__triglycerides_float_value": "Triglycerides",
    "bt__total_cholesterol_float_value": "Total cholesterol",
    "bt__hdl_cholesterol_float_value": "HDL cholesterol",
    "bt__ldl_cholesterol_float_value": "LDL cholesterol",
    "bt__non_hdl_cholesterol_float_value": "Non-HDL cholesterol",
    "bt__glucose_float_value": "Glucose",
    "bt__hba1c_float_value": "HbA1c",
    "bt__creatinine_float_value": "Creatinine",
    "bt__urate_float_value": "Urate",
    "bt__alt_float_value": "ALT",
    "bt__ast_float_value": "AST",
    "bt__ggt_float_value": "GGT",
}

FEATURE_LABELS = {
    "basic_nutrimatch": "Basic NutriMatch",
    "denovo_enriched": "De novo enriched",
    "nutrimatch_enhanced": "NutriMatch enhanced",
    "denovo_cardiometabolic": "De novo cardiometabolic",
    "denovo_broad_diet_health": "De novo broad diet-health",
    "denovo_microbiome": "De novo microbiome-oriented",
    "denovo_chemical_metabolomics": "De novo chemical/metabolomics",
    "denovo_mental_health": "De novo mental-health",
    "nutrimatch_cardiometabolic": "NutriMatch cardiometabolic",
    "nutrimatch_broad_diet_health": "NutriMatch broad diet-health",
    "nutrimatch_microbiome": "NutriMatch microbiome-oriented",
    "nutrimatch_chemical_metabolomics": "NutriMatch chemical/metabolomics",
    "nutrimatch_mental_health": "NutriMatch mental-health",
    "denovo_food_card_embedding": "Food-card embedding",
}

FEATURE_GROUPS = {
    "basic_nutrimatch": "baseline",
    "denovo_enriched": "enhanced table",
    "nutrimatch_enhanced": "enhanced table",
    "denovo_cardiometabolic": "downstream enhanced",
    "denovo_broad_diet_health": "downstream enhanced",
    "denovo_microbiome": "downstream enhanced",
    "denovo_chemical_metabolomics": "downstream enhanced",
    "denovo_mental_health": "downstream enhanced",
    "nutrimatch_cardiometabolic": "downstream enhanced",
    "nutrimatch_broad_diet_health": "downstream enhanced",
    "nutrimatch_microbiome": "downstream enhanced",
    "nutrimatch_chemical_metabolomics": "downstream enhanced",
    "nutrimatch_mental_health": "downstream enhanced",
    "denovo_food_card_embedding": "embedding enhanced",
}

df["target_label"] = df["target"].map(TARGET_LABELS).fillna(df["target"])
df["feature_label"] = df["feature_set"].map(FEATURE_LABELS).fillna(df["feature_set"])
df["feature_group"] = df["feature_set"].map(FEATURE_GROUPS).fillna("other")
df.head()

## Build Delta Table vs Basic NutriMatch

## Sanity Checks For Duplicate Inputs Or Results

Exact duplicate scores are often a sign that two feature sets are backed by the same source file or that their generated participant-level X matrices are identical. This section makes those cases visible before interpreting the plots.

In [ ]:
import hashlib
import json

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

config_path = PROJECT_ROOT / "downstream_analysis/tasks/cvd/all_diet_versions_config.json"
source_hash_rows = []

if config_path.exists():
    config = json.loads(config_path.read_text())
    for fs in config.get("feature_sets", []):
        source_path = PROJECT_ROOT / fs["path"]
        if source_path.exists():
            source_hash_rows.append({
                "feature_set": fs["name"],
                "source_path": str(source_path),
                "source_sha256": sha256_file(source_path),
                "source_bytes": source_path.stat().st_size,
            })

source_hashes = pd.DataFrame(source_hash_rows)
if source_hashes.empty:
    print("No source feature config/files found for source-hash checks.")
else:
    duplicate_sources = (
        source_hashes.groupby("source_sha256")
        .filter(lambda g: len(g) > 1)
        .sort_values(["source_sha256", "feature_set"])
    )
    source_hashes.to_csv(OUT_DIR / "source_feature_file_hashes.csv", index=False)
    duplicate_sources.to_csv(OUT_DIR / "duplicate_source_feature_files.csv", index=False)
    print("Wrote source hash audit files.")
    display(duplicate_sources if not duplicate_sources.empty else source_hashes)

x_hash_rows = []
for feature_set in sorted(df["feature_set"].dropna().unique()):
    fs_dir = PROJECT_ROOT / "downstream_analysis/tasks/cvd/outputs" / feature_set
    candidates = sorted(fs_dir.glob(f"X_{feature_set}_participant.*"))
    for x_path in candidates:
        x_hash_rows.append({
            "feature_set": feature_set,
            "x_path": str(x_path),
            "x_sha256": sha256_file(x_path),
            "x_bytes": x_path.stat().st_size,
        })

x_hashes = pd.DataFrame(x_hash_rows)
if x_hashes.empty:
    print("No participant X files found for X-hash checks.")
else:
    duplicate_x = (
        x_hashes.groupby("x_sha256")
        .filter(lambda g: len(g) > 1)
        .sort_values(["x_sha256", "feature_set"])
    )
    x_hashes.to_csv(OUT_DIR / "participant_x_file_hashes.csv", index=False)
    duplicate_x.to_csv(OUT_DIR / "duplicate_participant_x_files.csv", index=False)
    print("Wrote participant X hash audit files.")
    display(duplicate_x if not duplicate_x.empty else x_hashes)

metric_cols = [c for c in ["r2_mean", "rmse_mean", "pearson_r_mean"] if c in df.columns]
metric_key = ["target"] + (["model"] if "model" in df.columns else []) + metric_cols
duplicate_metrics = (
    df.groupby(metric_key, dropna=False)["feature_set"]
    .agg(lambda s: ", ".join(sorted(map(str, s))))
    .reset_index(name="feature_sets_with_same_metrics")
)
duplicate_metrics["n_feature_sets"] = duplicate_metrics["feature_sets_with_same_metrics"].str.count(",") + 1
duplicate_metrics = duplicate_metrics[duplicate_metrics["n_feature_sets"] > 1]
duplicate_metrics.to_csv(OUT_DIR / "duplicate_metric_rows.csv", index=False)
print("Wrote duplicate metric audit:", OUT_DIR / "duplicate_metric_rows.csv")
display(duplicate_metrics.head(50))

In [ ]:
baseline = df[df["feature_set"].eq("basic_nutrimatch")].copy()
if baseline.empty:
    raise ValueError("No basic_nutrimatch baseline rows found in the comparison CSV.")

keys = ["target"]
if "model" in df.columns:
    keys.append("model")

baseline = baseline[keys + ["r2_mean", "rmse_mean", "pearson_r_mean"]].rename(columns={
    "r2_mean": "baseline_r2",
    "rmse_mean": "baseline_rmse",
    "pearson_r_mean": "baseline_pearson_r",
})

compare = df.merge(baseline, on=keys, how="left")
compare = compare[~compare["feature_set"].eq("basic_nutrimatch")].copy()

compare["delta_r2"] = compare["r2_mean"] - compare["baseline_r2"]
compare["delta_pearson_r"] = compare["pearson_r_mean"] - compare["baseline_pearson_r"]
compare["rmse_improvement"] = compare["baseline_rmse"] - compare["rmse_mean"]
compare["improved_r2"] = compare["delta_r2"] > 0
compare["improved_rmse"] = compare["rmse_improvement"] > 0

compare_path = OUT_DIR / "basic_nutrimatch_vs_all_target_level.csv"
compare.to_csv(compare_path, index=False)
print("Wrote:", compare_path)

compare[["target_label", "feature_label", "feature_group", "baseline_r2", "r2_mean", "delta_r2", "baseline_rmse", "rmse_mean", "rmse_improvement"]].sort_values("delta_r2", ascending=False).head(30)

## Which Feature Sets Beat Basic NutriMatch Most Often?

In [ ]:
feature_summary = (
    compare.groupby(["feature_set", "feature_label", "feature_group"], dropna=False)
    .agg(
        mean_delta_r2=("delta_r2", "mean"),
        median_delta_r2=("delta_r2", "median"),
        best_delta_r2=("delta_r2", "max"),
        worst_delta_r2=("delta_r2", "min"),
        improved_targets=("improved_r2", "sum"),
        tested_targets=("target", "nunique"),
        mean_rmse_improvement=("rmse_improvement", "mean"),
    )
    .reset_index()
)
feature_summary["improved_target_fraction"] = feature_summary["improved_targets"] / feature_summary["tested_targets"]
feature_summary = feature_summary.sort_values(["mean_delta_r2", "improved_target_fraction"], ascending=False)

summary_path = OUT_DIR / "basic_nutrimatch_vs_all_feature_summary.csv"
feature_summary.to_csv(summary_path, index=False)
print("Wrote:", summary_path)
feature_summary

## Best Examples

In [ ]:
best_examples = compare.sort_values("delta_r2", ascending=False).head(20).copy()
best_examples["comparison"] = best_examples["target_label"] + " | " + best_examples["feature_label"]

best_path = OUT_DIR / "best_examples_vs_basic_nutrimatch.csv"
best_examples.to_csv(best_path, index=False)
print("Wrote:", best_path)

best_examples[["target_label", "feature_label", "feature_group", "baseline_r2", "r2_mean", "delta_r2", "baseline_rmse", "rmse_mean", "rmse_improvement", "aligned_rows", "feature_count"]]

## Plot: Mean Improvement by Feature Set

In [ ]:
plot_summary = feature_summary.sort_values("mean_delta_r2", ascending=True)

fig, ax = plt.subplots(figsize=(11, max(5, 0.42 * len(plot_summary))))
sns.barplot(
    data=plot_summary,
    x="mean_delta_r2",
    y="feature_label",
    hue="feature_group",
    dodge=False,
    ax=ax,
)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Mean R2 improvement over Basic NutriMatch")
ax.set_xlabel("Mean delta R2")
ax.set_ylabel("")
ax.legend(title="Feature group", loc="lower right")
plt.tight_layout()

path = OUT_DIR / "mean_delta_r2_vs_basic_nutrimatch.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
print("Wrote:", path)

## Plot: Target-by-Target Delta Heatmap

In [ ]:
heat = compare.copy()
feature_order = (
    heat.groupby("feature_label")["delta_r2"]
    .mean()
    .sort_values(ascending=False)
    .index
)

target_order = (
    heat.groupby("target_label")["delta_r2"]
    .max()
    .sort_values(ascending=False)
    .index
)

pivot = heat.pivot_table(index="feature_label", columns="target_label", values="delta_r2", aggfunc="mean")
pivot = pivot.reindex(index=feature_order, columns=target_order)

fig, ax = plt.subplots(figsize=(14, max(5, 0.42 * len(pivot))))
sns.heatmap(
    pivot,
    center=0,
    cmap="RdBu_r",
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Delta R2 vs Basic NutriMatch"},
    ax=ax,
)
ax.set_title("Every feature set compared with Basic NutriMatch")
ax.set_xlabel("Prediction target")
ax.set_ylabel("")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()

path = OUT_DIR / "target_by_target_delta_r2_heatmap.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
print("Wrote:", path)

## Plot: Best Thesis Examples

In [ ]:
top = best_examples.sort_values("delta_r2", ascending=True).copy()

fig, ax = plt.subplots(figsize=(12, max(5, 0.48 * len(top))))
sns.barplot(
    data=top,
    x="delta_r2",
    y="comparison",
    hue="feature_group",
    dodge=False,
    ax=ax,
)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Best examples where enhanced data beats Basic NutriMatch")
ax.set_xlabel("Delta R2 vs Basic NutriMatch")
ax.set_ylabel("")
ax.legend(title="Feature group", loc="lower right")
plt.tight_layout()

path = OUT_DIR / "best_examples_vs_basic_nutrimatch.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
print("Wrote:", path)

## Plot: Baseline vs Best Alternative Per Target

In [ ]:
best_per_target = (
    compare.sort_values("r2_mean", ascending=False)
    .drop_duplicates("target")
    .sort_values("delta_r2", ascending=False)
    .copy()
)

best_per_target_path = OUT_DIR / "best_alternative_per_target_vs_basic_nutrimatch.csv"
best_per_target.to_csv(best_per_target_path, index=False)
print("Wrote:", best_per_target_path)

fig, ax = plt.subplots(figsize=(8, 7))
sns.scatterplot(
    data=best_per_target,
    x="baseline_r2",
    y="r2_mean",
    hue="feature_group",
    s=90,
    ax=ax,
)

lo = np.nanmin([best_per_target["baseline_r2"].min(), best_per_target["r2_mean"].min(), 0])
hi = np.nanmax([best_per_target["baseline_r2"].max(), best_per_target["r2_mean"].max(), 0])
pad = (hi - lo) * 0.08 if hi > lo else 0.1
lo -= pad
hi += pad

ax.plot([lo, hi], [lo, hi], color="black", linestyle="--", linewidth=1)
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_title("Best alternative per target vs Basic NutriMatch")
ax.set_xlabel("Basic NutriMatch R2")
ax.set_ylabel("Best alternative R2")

for _, row in best_per_target.iterrows():
    ax.annotate(row["target_label"], (row["baseline_r2"], row["r2_mean"]), xytext=(4, 4), textcoords="offset points", fontsize=8)

plt.tight_layout()
path = OUT_DIR / "best_alternative_per_target_scatter.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
print("Wrote:", path)

best_per_target[["target_label", "feature_label", "feature_group", "baseline_r2", "r2_mean", "delta_r2", "baseline_rmse", "rmse_mean", "rmse_improvement"]]

## One-Line Result Summary

In [ ]:
n_targets = compare["target"].nunique()
n_feature_sets = compare["feature_set"].nunique()
positive_rows = int(compare["improved_r2"].sum())
total_rows = int(compare["improved_r2"].count())
best_row = compare.sort_values("delta_r2", ascending=False).iloc[0]

print(f"Compared Basic NutriMatch against {n_feature_sets} alternatives across {n_targets} targets.")
print(f"Enhanced/alternative feature sets improved R2 in {positive_rows}/{total_rows} target-feature comparisons.")
print(
    "Strongest example: "
    f"{best_row['feature_label']} predicting {best_row['target_label']} "
    f"improved R2 by {best_row['delta_r2']:.4f} "
    f"over Basic NutriMatch."
)